In [2]:
import pandas as pd
import numpy as np

# ================================
# STEP 1: LOAD DATA
# ================================
df = pd.read_excel(r"C:\Users\Path Bhimani\OneDrive\Desktop\Project Files\Forecasted table.xlsx")

# Change these if your column names differ
models = ['SARIMA', 'LR', 'RF']

# ================================
# STEP 2: SAFE ACTUAL (FOR MAPE)
# ================================
df['Actual_safe'] = df['Actual'].replace(0, np.nan)

# ================================
# STEP 3: ROW-WISE ERRORS
# ================================
for model in models:
    df[f'Error_{model}'] = df[model] - df['Actual']
    df[f'AbsError_{model}'] = np.abs(df[f'Error_{model}'])
    
    # FIXED MAPE (no inf)
    df[f'MAPE_{model}'] = np.abs(
        (df[model] - df['Actual']) / df['Actual_safe']
    ) * 100

# ================================
# STEP 4: ADD DAY COLUMN (96 STEPS/DAY)
# ================================
df['Day'] = (np.arange(len(df)) // 96) + 1

# ================================
# STEP 5: DAILY METRICS
# ================================
daily_results = []

for day, group in df.groupby('Day'):
    result = {'Day': day}
    
    actual = group['Actual'].values
    actual_safe = np.where(actual == 0, np.nan, actual)
    
    for model in models:
        pred = group[model].values
        error = actual - pred
        
        mae = np.mean(np.abs(error))
        rmse = np.sqrt(np.mean(error**2))
        
        # FIXED MAPE
        mape = np.nanmean(np.abs(error / actual_safe) * 100)
        
        sst = np.sum((actual - np.mean(actual))**2)
        sse = np.sum(error**2)
        r2 = 1 - (sse / sst) if sst != 0 else np.nan
        
        result[f'MAE_{model}'] = mae
        result[f'RMSE_{model}'] = rmse
        result[f'MAPE_{model}'] = mape
        result[f'R2_{model}'] = r2
    
    daily_results.append(result)

daily_df = pd.DataFrame(daily_results)

# ================================
# STEP 6: OVERALL METRICS (31 DAYS)
# ================================
overall_results = {}

actual_all = df['Actual'].values
actual_safe_all = np.where(actual_all == 0, np.nan, actual_all)

for model in models:
    pred_all = df[model].values
    error_all = actual_all - pred_all
    
    mae = np.mean(np.abs(error_all))
    rmse = np.sqrt(np.mean(error_all**2))
    
    # FIXED MAPE
    mape = np.nanmean(np.abs(error_all / actual_safe_all) * 100)
    
    sst = np.sum((actual_all - np.mean(actual_all))**2)
    sse = np.sum(error_all**2)
    r2 = 1 - (sse / sst)
    
    overall_results[f'MAE_{model}'] = mae
    overall_results[f'RMSE_{model}'] = rmse
    overall_results[f'MAPE_{model}'] = mape
    overall_results[f'R2_{model}'] = r2

overall_df = pd.DataFrame([overall_results])


df = df.round(2)
daily_df = daily_df.round(2)
overall_df = overall_df.round(4)

# ================================
# STEP 8: SAVE TO EXCEL
# ================================
with pd.ExcelWriter('Final_Error_Analysis.xlsx') as writer:
    df.to_excel(writer, sheet_name='Row_Data', index=False)
    daily_df.to_excel(writer, sheet_name='Daily_Metrics', index=False)
    overall_df.to_excel(writer, sheet_name='Overall_Metrics', index=False)
